# PINN4SOH 模块 3: PINN 前向传播

> 本手册严格按照 `src/03_model.py` 的**代码书写顺序**逐段解读。
> 同时标注原始代码 `Model/Model.py` 行号, 方便来回对照。
> 
> 每个代码块解释六件事: **代码 -> 逐行详解 -> 语法速查 -> 逻辑 -> 物理含义 -> 踩坑记录**。

---

### 数据流中的位置

```
.mat --> [模块1: 特征提取] --> CSV --> [模块2: 数据预处理] --> DataLoader
                                                               |
                                                               v
                                                      [模块3: 本模块]
                                                               |
                                                    +----------+----------+
                                                    |          |          |
                                                    v          v          v
                                               F(.): Encoder  G(.): 退化   autograd
                                               + Predictor   动力学方程   自动微分
                                               = Solution_u  = dynamical_F = u_t,u_x
                                                               |
                                                               v
                                                      [模块4: Loss 设计]
```

---


## 复现运行与源码对齐说明

> 当前 clean 代码：`src/03_model.py`  
> 原始代码：`Model/Model.py`  
> 执行规范：paper-reproduction；原始仓库只读。  
> 本手册已完成 Notebook JSON 与代码单元静态检查。涉及数据的单元需按 `DATA.md` 准备数据后，从头运行以生成本机结果。

本手册保留六层学习结构：纯净代码、逐行详解、语法表、数据流角色、物理含义与真实数据图、踩坑记录。论文与源码不一致处以源码为复现基准并单独说明。

### 背景: 为什么 PINN 需要物理约束?

**纯数据驱动方法的局限**: 普通神经网络只拟合"输入 -> 输出"的映射关系。对于电池 SOH 预测, 它只看到 17 维特征向量和 SOH 标签之间的统计相关性, 完全不知道电池退化是一个**受物理规律支配的动力学过程**。

**PINN 的核心思想**: 在神经网络中嵌入偏微分方程(PDE)形式的物理知识 -- 电池退化 SOH 满足一个未知的退化方程:

$$\frac{\partial u}{\partial t} = \mathcal{G}(x, t, u, \nabla u)$$

其中 $u$ = SOH(健康状态), $t$ = 循环编号(时间), $x$ = 16 维特征向量, $\mathcal{G}$ 是需要从数据中学习的退化动力学函数。

**模块 3 做什么**: 定义 PINN 的前向传播 -- 输入一个 batch 的配对数据, 输出 SOH 预测值 $u$ 和 PDE 残差 $f = u_t - \mathcal{G}$。这是后续计算三项 Loss 的基础。

---


### 模块 3 核心概念概览

> 行号记录自手册编写版本，后续整理可能产生偏移；核对实现时请以函数名和当前 `src/` 文件为准。

| 子模块 | clean_code 行号 | 原始代码行号 | 核心组件 |
|--------|:-----------:|:----------:|----------|
| 3a: Sin 激活 + MLP | L14-L55 | L10-L48 | 用 sin 代替 ReLU 的多层感知机 |
| 3b: Predictor + Solution_u | L58-L98 | L51-L86 | Encoder(17->32) + Predictor(32->1) |
| 3c: PINN 模型组装 | L101-L118 | L127-L162 | Solution_u + dynamical_F 双模型 |
| 3d: PINN.forward() [核心] | L120-L139 | L216-L235 | autograd 求 u_t/u_x + PDE 残差 |
| 3e: predict / Test / Valid | L141-L172 | L183-L214 | 评估模式下的推理 |
| 主程序验证 | L175-L212 | -- | 模型实例化 + forward 验证 |

### 目录

| 章节 | clean_code 行号 | 原始代码行号 | 内容 |
|------|:----------:|:----------:|------|
| [1. 3a: Sin 激活 + MLP](#sec3a) | 14-55 | 10-48 | `Sin()` + `MLP` 类 |
| [2. 3b: Predictor + Solution_u](#sec3b) | 58-98 | 51-86 | `Predictor` + `Solution_u` 类 |
| [3. 3c: PINN 模型组装](#sec3c) | 101-118 | 127-162 | `PINN.__init__()` 双模型初始化 |
| [4. 3d: PINN.forward()（核心）](#sec3d) | 120-139 | 216-235 | autograd 自动微分 + PDE 残差 |
| [5. 3e: predict / Test / Valid](#sec3e) | 141-172 | 183-214 | 推理与评估 |
| [6. 主程序验证](#sec6) | 175-212 | -- | 完整模型架构验证 |

---


<a id="sec3a"></a>
## 1. 3a: Sin 激活函数 + MLP

> **对应 src/03_model.py 第 14-55 行**
> **对应原始代码 Model/Model.py 第 10-48 行**

一句话: 定义 PINN 的基础构建块 -- `Sin()`(正弦激活函数)和 `MLP`(支持 Sin + Xavier 初始化 + Dropout 的通用多层感知机构造器)。


### 1.1 代码块


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.autograd import grad

device = 'cuda' if torch.cuda.is_available() else 'cpu'


class Sin(nn.Module):
    """Sin 激活函数"""
    def __init__(self):
        super(Sin, self).__init__()

    def forward(self, x):
        return torch.sin(x)


class MLP(nn.Module):
    """多层感知机：Sin 激活 + Xavier 初始化 + Dropout 正则化"""
    def __init__(self, input_dim=17, output_dim=1, layers_num=4,
                 hidden_dim=50, dropout=0.2):
        super(MLP, self).__init__()
        assert layers_num >= 2, "layers must be greater than 2"
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layers_num = layers_num
        self.hidden_dim = hidden_dim

        self.layers = []
        for i in range(layers_num):
            if i == 0:
                self.layers.append(nn.Linear(input_dim, hidden_dim))
                self.layers.append(Sin())
            elif i == layers_num - 1:
                self.layers.append(nn.Linear(hidden_dim, output_dim))
            else:
                self.layers.append(nn.Linear(hidden_dim, hidden_dim))
                self.layers.append(Sin())
                self.layers.append(nn.Dropout(p=dropout))
        self.net = nn.Sequential(*self.layers)
        self._init()

    def _init(self):
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)

    def forward(self, x):
        x = self.net(x)
        return x


### 1.2 🔍 逐行详解: Sin 激活 + MLP 构造器

下面把 3a 的代码逐行拆开，标注每一行的**结构角色**（函数定义？变量赋值？循环？）和**具体功能**。理解代码结构是理解逻辑的前提。

---

**【模块 A】导入依赖库 + 设备配置**

```python
import torch                                  # [导入模块] PyTorch 深度学习框架
    #  └──┬──┘                                    torch 是整个 PyTorch 的顶层包
    #    模块名                                     提供 Tensor, autograd, nn 等核心功能

import torch.nn as nn                         # [导入模块 + 别名] torch.nn 子包 -> 别名 nn
    #  └───┬───┘ └┬┘ └┘                             torch.nn: 神经网络组件库
    #   子模块路径  as 别名                           nn.Linear, nn.Module, nn.Sequential 等
    #                                               使用别名后写 nn.Linear 而非 torch.nn.Linear

from torch.autograd import grad               # [从模块导入特定函数] grad = 自动微分引擎
    #  └──────┬──────┘        └┬┘
    #  torch.autograd 子包     grad 函数
    #
    #  grad(output, input) 计算标量 output 对 input 的偏导数
    #  PINN 用它求 du/dt (退化速率) 和 du/dx (特征敏感度)
```

```python
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#       [条件表达式: GPU 可用? -> 'cuda' : 'cpu']
#
#  逐段拆解:
#  torch.cuda.is_available() -> 返回 bool: 检测是否有 NVIDIA GPU
#  'cuda' if True else 'cpu' -> 三元表达式: GPU可用用cuda, 否则用cpu
#  后续所有 model.to(device) 和 tensor.to(device) 都引用此变量
```

> **关键区分**: `import X` 导入整个模块，使用时需 `X.func()`；`from X import Y` 只导入 Y，可直接用 `Y()`。

> **为什么 PINN 需要 grad?** 普通分类/回归网络只需要 forward 得到预测值。PINN 的 PDE Loss 需要 $u_t = \partial u/\partial t$ 和 $u_x = \partial u/\partial x$ -- 这些偏导数不是手动计算的，是 `autograd.grad()` 自动求出来的。

---

**【模块 B】Sin 激活函数类 —— PINN 区别于普通网络的第一个标志**

```python
class Sin(nn.Module):                          # [类定义头]
    #  └┬┘ └───┬───┘                              class: 定义类的关键字
    #   类名  父类 nn.Module                       Sin: 类名（见名知义 = 正弦函数）
    #                                             继承 nn.Module 才能被 PyTorch 追踪参数

    def __init__(self):                        # [构造函数 __init__]
        super(Sin, self).__init__()            # [调用父类构造函数]
        #  └──┬──┘                                super(): 获取父类 nn.Module
        #   调用父类                                .__init__(): 执行父类的初始化
        #
        #  为什么 Sin 的 __init__ 是空的？
        #  Sin 没有可训练参数——它只是一个纯数学函数 sin(x)
        #  不需要定义 weight 或 bias
        #  但 super().__init__() 必须调用 —— 否则 nn.Module 的内部注册机制不工作

    def forward(self, x):                      # [前向传播定义]
        #  └──┬──┘ └┘                              forward: nn.Module 的约定方法名
        #   方法名   形参 x                           重写父类的 forward, 定义"输入->输出"
        #                                           x: 任意形状的 Tensor
        return torch.sin(x)                    # [返回语句 return]
        #       └───┬───┘                           torch.sin(): 逐元素正弦
        #     PyTorch 的正弦函数                      对输入 Tensor 中每个元素计算 sin
        #
        #  为什么用 torch.sin 而不是 math.sin?
        #  torch.sin 是 PyTorch 的张量操作:
        #  - 支持 GPU 加速
        #  - 支持 autograd 自动求导
        #  - 支持 broadcasting 广播
        #  math.sin 只能处理单个 Python 数字
```

> **函数整体角色**: `Sin` 是 PINN 区别于普通神经网络的第一个标志。它把 `nn.ReLU()` 替换为 `torch.sin()` —— 因为 PINN 需要高阶导数，而 sin 的任意阶导数都存在且是周期性的（$\sin' = \cos$, $\sin'' = -\sin$, $\sin''' = -\cos$, ... 循环往复）。ReLU 的二阶导数处处为 0，autograd 无法传播 PDE 物理信息。

---

**【模块 C】MLP 类 —— 通用多层感知机构造器（工厂模式）**

```python
class MLP(nn.Module):                          # [类定义头]
    #  与 Sin 一样继承 nn.Module
    #  MLP = Multi-Layer Perceptron = 多层感知机
```

```python
    def __init__(self, input_dim=17, output_dim=1, layers_num=4,
    #  [构造函数]   [参数1: 输入维度] [参数2: 输出维度] [参数3: 总层数]
                 hidden_dim=50, dropout=0.2):
    #            [参数4: 隐藏层宽度] [参数5: Dropout概率]
    #
    #  逐参数说明:
    #  input_dim=17:   输入特征数 = 16个统计特征 + 1个cycle_index
    #                 默认值 17 —— 论文中 Solution_u 的输入
    #  output_dim=1:   输出维度 = SOH 的标量预测值
    #                 默认值 1 —— 回归问题的标准输出
    #  layers_num=4:   总层数(含输入层和输出层)
    #                 例如 4 层 = 1个输入层 + 2个隐藏层 + 1个输出层
    #  hidden_dim=50:  每个隐藏层的神经元数
    #                 决定了模型的"宽度" —— 越大越能表达复杂函数
    #  dropout=0.2:    训练时随机丢弃 20% 神经元
    #                 防止过拟合 —— 小数据集上特别重要
```

```python
        super(MLP, self).__init__()            # [调用父类构造函数]
        assert layers_num >= 2, "layers must be greater than 2"
        #      [断言检查: 层数必须 >=2]
        #
        #  assert: Python 的运行时检查，条件为 False 时抛 AssertionError
        #  为什么 layers_num 必须 >= 2?
        #  至少需要输入层 + 输出层 = 2 层
        #  如果 layers_num=1，就没有隐藏层 → 退化为线性回归
```

```python
        self.input_dim = input_dim             # [变量赋值] 保存输入维度
        self.output_dim = output_dim           # [变量赋值] 保存输出维度
        self.layers_num = layers_num           # [变量赋值] 保存层数
        self.hidden_dim = hidden_dim           # [变量赋值] 保存隐藏层宽度
        #  为什么把这 4 个参数存为实例属性?
        #  方便外部查询 model.input_dim、model.layers_num
        #  虽然 forward() 不直接用它们，但调试和文档时很有用
```

```python
        self.layers = []                       # [变量初始化] 空列表，逐层构建后放入
        #  为什么用列表而不是直接写 self.net = Sequential(...)?
        #  因为层数是动态的(layers_num 可变) —— 无法用固定代码
        #  先用 for 循环填充列表，再一次性转成 Sequential
```

```python
        for i in range(layers_num):            # [循环语句 for] i = 0, 1, 2, ..., layers_num-1
        #  └┬┘ └──────┬──────┘                   range(N) 生成 [0, 1, ..., N-1]
        #   循环变量   迭代范围                      每次迭代构建一层
```

```python
            if i == 0:                         # [条件判断] 第一层 = 输入层
                self.layers.append(nn.Linear(input_dim, hidden_dim))
                #               [全连接层] input_dim -> hidden_dim
                #
                #  nn.Linear(in, out) 的参数数量: in*out + out
                #  以 layers_num=4, input_dim=17, hidden_dim=50 为例:
                #  17*50 + 50 = 900 个参数 (= 850权重 + 50偏置)
                #
                self.layers.append(Sin())      # [实例化 Sin 激活]
                #  Linear 之后紧跟激活函数 —— 神经网络的标配
                #  注意: 输入层之后也有激活函数(和隐藏层一样)
```

```python
            elif i == layers_num - 1:          # [条件分支] 最后一层 = 输出层
                self.layers.append(nn.Linear(hidden_dim, output_dim))
                #               [全连接层] hidden_dim -> output_dim
                #
                #  参数数量: 50*1 + 1 = 51 (50权重 + 1偏置)
                #
                #  【关键】输出层为什么没有 Sin 激活?
                #  SOH 预测值 ∈ [0.8, 1.0] —— 不在 [-1, 1] 区间
                #  如果加 Sin，输出被限制在 [-1, 1] —— 预测范围受限
                #  输出层不加激活 = 线性输出 = 任意实数范围
```

```python
            else:                              # [其他层] 中间隐藏层
                self.layers.append(nn.Linear(hidden_dim, hidden_dim))
                #               [全连接层] hidden_dim -> hidden_dim
                #               参数: 50*50 + 50 = 2550
                self.layers.append(Sin())      # [Sin 激活]
                self.layers.append(nn.Dropout(p=dropout))
                #               [Dropout 正则化层] [丢弃概率=0.2]
                #
                #  Dropout 机制: 训练时随机把 20% 的神经元输出置 0
                #  相当于每次训练都在一个"随机子网络"上进行
                #  效果: 防止神经元之间共适应 → 减少过拟合
                #  评估时 (model.eval()): Dropout 自动关闭, 所有神经元参与
```

```python
        self.net = nn.Sequential(*self.layers)
        #  └─┬─┘          └─────┬─────┘
        #  [nn.Sequential]  [* 解包运算符]
        #
        #  逐段拆解:
        #  self.layers     → 列表, 如 [Linear, Sin, Linear, Sin, Dropout, ...]
        #  *self.layers    → 解包: 把列表元素变成 Sequential 的独立参数
        #                     Sequential(Linear, Sin, Linear, Sin, ...)
        #  不用 * 直接传:  Sequential([Linear, Sin, ...]) → 报错!
        #  nn.Sequential   → 按顺序串联所有层, input -> layer0 -> layer1 -> ... -> output
        #
        #  为什么最后才转 Sequential?
        #  因为列表方便逐个添加(append)，Sequential 创建后不能修改
        self._init()                           # [函数调用] 执行参数初始化
        #  在 Sequential 创建好之后, 立即对所有 Linear 层做 Xavier 初始化
```

```python
    def _init(self):                           # [私有方法] _开头 = 约定为"内部使用"
        #  └┬┘                                    不是 Python 的强制语法，是编码约定
        #   方法名                                   外部不应直接调用此方法
        for layer in self.net:                 # [循环语句] 遍历 Sequential 中的每一层
            if isinstance(layer, nn.Linear):   # [类型检查 isinstance]
            #  └────┬────┘ └───┘ └────┬────┘      isinstance(obj, Type): 检查 obj 是不是 Type 的实例
            #   内置函数     层对象    目标类型        这里: 只对 nn.Linear 层做初始化
                nn.init.xavier_normal_(layer.weight)
                #  └──────┬──────┘ └─────┬─────┘
                #  nn.init 子模块      layer 的权重矩阵
                #
                #  Xavier 正态初始化:
                #  权重 W ~ N(0, sqrt(2 / (fan_in + fan_out)))
                #  fan_in = 该层的输入神经元数
                #  fan_out = 该层的输出神经元数
                #
                #  为什么用 Xavier?
                #  它保持每层输出的方差大致相同 → 防止梯度爆炸/消失
                #  对 Sin 激活特别有效 —— Sin 的导数 cos(x) ∈ [-1,1]
                #  不会像 ReLU 那样让一半梯度消失
                #
                #  _init 不初始化 bias: bias 默认初始化为 0 (PyTorch 默认)
```

```python
    def forward(self, x):                      # [前向传播定义]
        x = self.net(x)                        # [核心计算] x -> Sequential -> 输出
        return x                               # [返回语句]
        #
        #  forward 只有 2 行 —— 因为所有层的逻辑已经在 Sequential 中封装好了
        #  典型的 PyTorch 风格: __init__ 构建计算图, forward 只调用
```

> **函数整体角色**: `MLP` 是 PINN 的**可复用组件工厂**。不是固定的网络结构, 而是一个构造器 —— 根据传入的参数动态生成不同层数和宽度的 MLP。`Solution_u` 的 Encoder 和 `dynamical_F` 都复用同一个 `MLP` 类, 只是传入不同的参数。

---

**【设计决策】为什么 MLP 用工厂模式而非固定网络?**

| 方案 | 做法 | 缺点 |
|------|------|------|
| 固定网络 | 为 Encoder 写 `class Encoder`, 为 dynamical_F 写 `class DynamicalF` | 代码重复, 修改一处需同步两处 |
| **工厂模式 (本实现)** | 一个 `MLP` 类, 通过参数控制结构 | ✅ 复用: Encoder=MLP(17->32), G=MLP(35->1), 改 Bug 只改一处 |

**【层数推导】以 layers_num=4 为例**

| i | 条件 | 添加的层 | 参数数量 |
|:-:|------|----------|:------:|
| 0 | i == 0 (输入层) | Linear(17,50) + Sin() | 17*50+50=900 |
| 1 | else (隐藏层) | Linear(50,50) + Sin() + Dropout(0.2) | 50*50+50=2550 |
| 2 | else (隐藏层) | Linear(50,50) + Sin() + Dropout(0.2) | 50*50+50=2550 |
| 3 | i == layers_num-1 (输出层) | Linear(50,1) | 50*1+1=51 |
| **合计** | | 7 层 (4 Linear + 2 Sin + 2 Dropout) | **6051** |

---

**【设计总结】Sin + MLP vs 普通 MLP (ReLU)**

| 特性 | 普通 MLP (ReLU) | PINN MLP (Sin) |
|------|:-----------:|:----------:|
| 激活函数 | ReLU(x) = max(0,x) | Sin(x) = sin(x) |
| 一阶导数 | 0 (x<0) 或 1 (x>0) | cos(x), 处处非零 |
| 二阶导数 | 处处为 0 | -sin(x), 存在且光滑 |
| 高阶导数 | 全为 0 | 周期性存在, 永远不会退化 |
| PINN 为什么选它 | 二阶导为零 → L_PDE 梯度消失 → 不收敛 | 任意阶导数非零 → autograd 在任何深度都有效 |


### 1.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `torch.sin(x)` | 逐元素正弦函数 | 任意阶导数存在且非零, autograd 求 u_t 和 u_x 有效 |
| `nn.Sequential(*list)` | 将列表解包后串联成顺序网络 | 动态构建层数: `*` 把 `[L1,L2,L3]` 变成 `Sequential(L1,L2,L3)` |
| `nn.init.xavier_normal_(w)` | Xavier 正态初始化(末尾 `_` = inplace) | 保持各层输出方差一致, 防止梯度爆炸/消失 |
| `assert layers_num >= 2` | 运行时条件检查 | 输入层+输出层至少 2 层, 防御性编程 |
| `super().__init__()` | 调用父类 nn.Module 的构造函数 | 不调用则模型参数无法被 PyTorch 注册追踪 |
| `isinstance(layer, nn.Linear)` | 运行时类型检查 | 只对 Linear 层初始化, 跳过 Sin/Dropout/Sequential |
| `nn.Dropout(p=0.2)` | 训练时随机置零 20% 神经元 | 防止过拟合, 相当于每次训练一个随机子网络 |


### 1.4 逻辑: MLP 在整体流程中的角色

```
输入 x (17维: 16特征 + cycle_index)
    |
    v MLP(layers_num=4)
    +-- Linear(17->50) + Sin()              <- 第0层: 升维 + 非线性
    +-- Linear(50->50) + Sin() + Dropout    <- 第1层: 特征变换 + 正则化
    +-- Linear(50->50) + Sin() + Dropout    <- 第2层: 特征变换 + 正则化
    +-- Linear(50->1)                        <- 第3层: 降维到1维标量
    |
    v
u (1维: SOH 预测值)
```

**MLP 被复用了两次 — 同一份代码, 不同参数:**
- `Solution_u.encoder = MLP(input_dim=17, output_dim=32, layers_num=3, hidden_dim=60)`
- `dynamical_F = MLP(input_dim=35, output_dim=1, layers_num=3, hidden_dim=60)`

注意两者的输入维度完全不同 (17 vs 35), 但都能用同一个 MLP 类构造 —— 这就是工厂模式的价值。


### 1.5 物理含义: Sin 为什么是 PINN 的关键

用真实数据直观对比 Sin vs ReLU 的表现:


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

x = torch.linspace(-3, 3, 300)

relu = torch.relu(x)
sin = torch.sin(x)

relu_grad = torch.relu(torch.ones_like(x))
sin_grad = torch.cos(x)
sin_grad2 = -torch.sin(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(x, relu, 'b-', linewidth=2, label='ReLU')
axes[0].plot(x, sin, 'r-', linewidth=2, label='Sin')
axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('Activation Function')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, relu_grad, 'b-', linewidth=2, label="ReLU'")
axes[1].plot(x, sin_grad, 'r-', linewidth=2, label="Sin' = cos(x)")
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[1].set_title('First Derivative')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, torch.zeros_like(x), 'b-', linewidth=2, label="ReLU'' = 0")
axes[2].plot(x, sin_grad2, 'r-', linewidth=2, label="Sin'' = -sin(x)")
axes[2].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[2].set_title('Second Derivative')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('ReLU vs Sin: Why PINN Must Use Sin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'../outputs/figures/model/sin_vs_relu.png', dpi=150, bbox_inches='tight')
plt.show()

print('ReLU second derivative: 0 everywhere -> autograd cannot propagate PDE Loss')
print('Sin  second derivative: -sin(x) != 0 -> autograd computes arbitrary-order derivatives')


### 1.6 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 把 Sin 替换成 ReLU | L_PDE 的梯度全为 0, 模型退化为纯数据驱动 MLP, 物理约束无效 | PINN 必须用 sin, 这是前提 |
| 忘记 `super().__init__()` | 模型参数 `.parameters()` 返回空列表, optimizer 找不到参数 | `__init__` 第一行必须调用 `super().__init__()` |
| dropout=0 (不丢弃) | 小数据集上严重过拟合: train loss 很低但 valid loss 很高 | 保留 dropout=0.2, 这是论文验证过的值 |
| `nn.Sequential(self.layers)` 没加 `*` | TypeError: list is not a Module | `nn.Sequential(*self.layers)` — 必须解包 |
| 输出层也加 Sin 激活 | SOH 预测值被限制在 [-1, 1], 但真实 SOH ∈ [0.8, 1.0], 始终预测不到 | 输出层不加激活, 保持线性输出 |

---


<a id="sec3b"></a>
## 2. 3b: Predictor + Solution_u

> **对应 src/03_model.py 第 58-98 行**
> **对应原始代码 Model/Model.py 第 51-86 行**

一句话: `Solution_u` = Encoder(MLP 17->32) + Predictor(32->1), 是 PINN 的 SOH 预测分支 —— 输入 (特征+cycle_index), 输出 SOH 预测值。


### 2.1 代码块


In [ ]:
class Predictor(nn.Module):
    """预测器：Dropout → Linear(40→32) → Sin → Linear(32→1)"""
    def __init__(self, input_dim=40):
        super(Predictor, self).__init__()
        self.net = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(input_dim, 32),
            Sin(),
            nn.Linear(32, 1)
        )
        self.input_dim = input_dim

    def forward(self, x):
        return self.net(x)


class Solution_u(nn.Module):
    """SOH 预测网络：Encoder(MLP 17→32) + Predictor(32→1)"""
    def __init__(self):
        super(Solution_u, self).__init__()
        self.encoder = MLP(input_dim=17, output_dim=32, layers_num=3,
                           hidden_dim=60, dropout=0.2)
        self.predictor = Predictor(input_dim=32)
        self._init_()

    def get_embedding(self, x):
        return self.encoder(x)

    def forward(self, x):
        x = self.encoder(x)
        x = self.predictor(x)
        return x

    def _init_(self):
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.constant_(layer.bias, 0)
            elif isinstance(layer, nn.Conv1d):
                nn.init.xavier_normal_(layer.weight)
                nn.init.constant_(layer.bias, 0)


### 2.2 🔍 逐行详解: Predictor + Solution_u

---

**【模块 A】Predictor 类 —— 从嵌入向量到 SOH 预测值**

```python
class Predictor(nn.Module):                    # [类定义头]
    #  输入: 编码后的嵌入向量 (32 维)
    #  输出: SOH 预测值 (1 维标量)
    #  角色: "解码器"——把抽象的嵌入空间映射回物理的 SOH 值
```

```python
    def __init__(self, input_dim=40):
    #  [构造函数]   [参数: 输入维度, 默认=40]
    #
    #  默认值 40 来自原始论文中 Encoder 输出维度的早期设定
    #  在本复现中, 实际调用时传入 input_dim=32 (Encoder 输出维度)
    #  所以默认值 40 不起实际作用 —— 每次调用都显式传参
```

```python
        super(Predictor, self).__init__()      # [调用父类构造函数]
        self.net = nn.Sequential(              # [Sequential 顺序容器]
            nn.Dropout(p=0.2),                 # [Dropout层] 正则化: 随机丢弃 20%
            #  └──────┬──────┘                    注意: Dropout 放在最前面 —— 先丢弃再进入 Linear
            #   nn.Dropout(p=0.2)                 这与 MLP 中放在 Sin 之后的顺序不同
            #                                       都是合理的, 效果大致相同
            nn.Linear(input_dim, 32),          # [全连接层] input_dim -> 32
            #  └────┬────┘ └──┬──┘ └┘             参数数量: input_dim*32 + 32
            #  nn.Linear  in_dim out_dim         当 input_dim=32: 32*32+32 = 1056
            Sin(),                             # [Sin 激活] 非线性变换
            #  与 MLP 类一致 —— 用 Sin 而非 ReLU
            nn.Linear(32, 1)                   # [全连接层] 32 -> 1
            #  参数数量: 32*1+1 = 33
            #  输出: SOH 的标量预测值
        )
        self.input_dim = input_dim             # [变量赋值] 保存输入维度供外部查询

    def forward(self, x):                      # [前向传播]
        return self.net(x)                     # [返回语句] 直接走 Sequential 管道
        #
        #  Predictor 的 forward 和 MLP 的 forward 一样简洁
        #  所有工作都在 __init__ 构造好的 Sequential 中完成
```

> **Predictor 的角色**: 从抽象嵌入空间 (32 维) 映射到物理空间 (1 维 SOH)。类比: Encoder 是"阅读理解"——从 17 个特征中提取含义, 压缩到 32 维嵌入; Predictor 是"回答问题"——从嵌入中读出 SOH 这个数。

**【Predictor 层结构可视化】**

```
输入: 嵌入向量 (32维)
    |
    v
Dropout(p=0.2)         <- 训练时随机屏蔽 20%
    |
    v
Linear(32, 32) + Sin()  <- 嵌入空间内的特征变换
    |
    v
Linear(32, 1)           <- 投射到标量: SOH ∈ (-∞, +∞)
    |
    v
输出: SOH 预测值 (1维)
```

---

**【模块 B】Solution_u 类 —— PINN 的 SOH 预测分支**

```python
class Solution_u(nn.Module):                   # [类定义头]
    #  └────┬────┘                                u = SOH 的数学符号 (PDE 中的未知函数)
    #    类名                                       见名知义: 这个类代表 PDE 的解 u(x,t)
    #                                              也就是我们想预测的 SOH
```

```python
    def __init__(self):
        super(Solution_u, self).__init__()

        self.encoder = MLP(input_dim=17, output_dim=32, layers_num=3,
        #              [实例化 MLP]  [17维输入]  [32维输出]  [3层]
                           hidden_dim=60, dropout=0.2)
        #                  [隐藏层宽度=60]  [Dropout率=0.2]
        #
        #  逐段拆解 Encoder 的 MLP 配置:
        #  input_dim=17:    16个统计特征 + 1个cycle_index
        #  output_dim=32:   嵌入向量的维度 —— 论文验证的最优值
        #  layers_num=3:    共3层 = 输入层(17->60) + 隐藏层(60->60) + 输出层(60->32)
        #  hidden_dim=60:   隐藏层宽度 —— 比 MLP 默认值(50)更宽
        #                   嵌入空间需要更多容量来表达 16 个特征的变化
        #  dropout=0.2:     正则化强度
        #
        #  Encoder 的参数数量:
        #  第0层: 17*60+60 = 1080
        #  第1层: 60*60+60 = 3660
        #  第2层: 60*32+32 = 1952
        #  Encoder 合计: 1080 + 3660 + 1952 = 6692

        self.predictor = Predictor(input_dim=32)
        #                [实例化 Predictor]  [接收32维嵌入]
        #
        #  Predictor 参数数量: 32*32+32 + 32*1+1 = 1056 + 33 = 1089
        #  Solution_u 总参数: 6692 + 1089 = 7781

        self._init_()                          # [函数调用] 执行参数初始化
        #  └──┬──┘                                注意方法名: _init_() 而非 _init()
        #   两个下划线后缀                         如果写成 _init(), SyntaxError:
        #                                         Python 的 __init__ 是保留方法名
        #                                         _init_() 是自定义的初始化方法
```

```python
    def get_embedding(self, x):                # [辅助方法] 获取中间嵌入
        return self.encoder(x)                 # [返回 Encoder 输出]
        #
        #  这个方法不改变模型状态 —— 只是 forward 的前半步
        #  用途: 调试时可视化嵌入空间 (如 PCA 降维图)
        #       迁移学习时复用 Encoder 的嵌入
        #  正常训练不需要调用此方法
```

```python
    def forward(self, x):                      # [前向传播定义]
        #  x: 输入张量, 形状 (batch, 17)
        #  17 = 16特征 + 1个cycle_index
        x = self.encoder(x)                    # 第一步: 编码 17维 -> 32维嵌入
        #  └─────┬─────┘                          形状变换: (batch, 17) -> (batch, 32)
        #  调用 MLP.forward(x)

        x = self.predictor(x)                  # 第二步: 预测 32维嵌入 -> 1维 SOH
        #  └───────┬───────┘                       形状变换: (batch, 32) -> (batch, 1)
        #  调用 Predictor.forward(x)

        return x                               # [返回语句]
        #  返回: (batch, 1) -> SOH 预测值
        #
        #  【形状追踪】
        #  (batch, 17) --encoder--> (batch, 32) --predictor--> (batch, 1)
        #  这就是 Solution_u 的完整数据流
```

```python
    def _init_(self):                          # [参数初始化方法]
        #  注意: 这是 _init_ (两个下划线后缀), 不是 __init__
        #  __init__ 是 Python 的构造函数, 不能覆盖
        for layer in self.modules():           # [遍历所有子模块]
        #  └──────┬──────┘                        self.modules(): 递归遍历所有子模块
        #  nn.Module 的方法                        包括 Encoder 和 Predictor 内部的所有层

            if isinstance(layer, nn.Linear):   # [类型检查] 只对全连接层做初始化
                nn.init.xavier_normal_(layer.weight)
                #  [Xavier 正态初始化] 与 MLP._init() 一致
                nn.init.constant_(layer.bias, 0)
                #  [bias 初始化为 0]
                #
                #  为什么 bias 初始化为 0?
                #  bias 代表"在没有任何输入时的基线输出"
                #  初始化为 0 是最安全的选择 —— 不引入先验偏移
                #  训练过程中 bias 会自动学习到合适的值

            elif isinstance(layer, nn.Conv1d): # [卷积层检查]
            #  防御性编程 —— 虽然当前设计没有 Conv1d
            #  但如果未来有人给 Solution_u 加卷积层
            #  这段代码仍然能正确初始化
                nn.init.xavier_normal_(layer.weight)
                nn.init.constant_(layer.bias, 0)
```

> **函数整体角色**: `Solution_u` 是 PINN 的预测分支 —— 给定带有 cycle_index 的输入向量 (17 维), 输出 SOH 预测值 (1 维)。Encoder 负责"理解"特征并压缩到 32 维嵌入, Predictor 负责从嵌入中"读出" SOH。两步走的设计让中间嵌入可以被复用和可视化。

---

**【设计决策】Solution_u 的关键选择**

| 决策 | 选项 | 选择 | 原因 |
|------|------|:---:|------|
| Encoder 输出维度 | 8 / 16 / **32** / 64 | 32 | 论文实验验证: 32 维嵌入能充分表达 16 特征的变化, 再大则冗余 |
| Encoder 层数 | 2 / **3** / 4 | 3 | 3 层 = 输入+1隐藏+输出, 对 17->32 的编码任务刚好 |
| Predictor 为什么是 Dropout -> Linear 而非 Linear -> Dropout? | 两种顺序 | 先 Dropout | 在进入 Linear 之前做 Dropout, 相当于随机屏蔽输入特征 —— 效果与 MLP 中在 Sin 后做 Dropout 类似 |
| _init_ vs __init__ 命名 | 覆盖 __init__ / 新方法 | 新方法 `_init_` | `__init__` 是 Python 保留方法, 覆盖会导致父类构造函数被跳过 |

---

**【形状追踪】Solution_u 完整数据流**

```
输入: x (batch, 17)
    |
    v  Encoder: MLP(17 -> 60 -> 60 -> 32)
    |  第0层: Linear(17,60) + Sin  →  参数 1080
    |  第1层: Linear(60,60) + Sin + Dropout  →  参数 3660
    |  第2层: Linear(60,32)  →  参数 1952
    |  Encoder 合计参数: 6692
    |
    v  嵌入: (batch, 32)
    |
    v  Predictor: Dropout -> Linear(32,32)+Sin -> Linear(32,1)
    |  Dropout  →  参数 0
    |  Linear(32,32)+Sin  →  参数 1056
    |  Linear(32,1)  →  参数 33
    |  Predictor 合计参数: 1089
    |
    v
输出: u (batch, 1) = SOH 预测值

Solution_u 总参数: 6692 + 1089 = 7781
```


### 2.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `nn.init.constant_(tensor, val)` | 将张量所有元素填充为 val | bias 初始化为 0, 不对前向传播引入偏移 |
| `self.modules()` | 递归遍历所有子模块(含子模块的子模块) | 确保 Encoder + Predictor 内部的所有 Linear 都被初始化 |
| `isinstance(obj, cls)` | 检查对象是否是某个类的实例 | 只对 Linear/Conv1d 层做初始化, 跳过 Sin/Dropout/Sequential |
| `get_embedding(x)` | 自定义方法: 只走 Encoder 的前半步 | 调试/可视化时查看嵌入空间结构, 不改变模型权重 |
| `nn.Linear(in, out)` 参数数 | `in*out + out` (权重 + 偏置) | 方便算总参数量和检查显存占用 |
| `_init_` vs `__init__` | 两个下划线后缀 vs 前后各两个 | `__init__` 被 Python 保留, 自定义初始化必须用其他名字 |


### 2.4 逻辑: Solution_u 在整体流程中的角色

```
输入 x (17维: 16特征 + cycle_index)
    |
    v Solution_u.forward(x)
    |
    +-- Encoder: MLP(17->60->60->32)
    |   参数量: 6,692
    |   输出: 32维嵌入向量 (抽象特征表示)
    |   每一个维度编码了特征的不同方面
    |
    +-- Predictor: Dropout -> Linear(32,32) + Sin -> Linear(32,1)
    |   参数量: 1,089
    |   输出: 1维 SOH 预测值
    |
    v
u = Solution_u(x)    <- PINN 的预测结果
     |
     +-- 参与 L_data = 0.5*MSE(u1, y1) + 0.5*MSE(u2, y2)
     +-- 参与 autograd: u_t = du/dt, u_x = du/dx (在 PINN.forward 中调用)
```


### 2.5 物理含义: Encoder 学到了什么?

用真实数据查看 Encoder 嵌入空间的降维可视化:


In [ ]:
import sys
from pathlib import Path
CLEAN = Path(r'../src')
sys.path.insert(0, str(CLEAN))
from module_loader import load_clean_module
runner = load_clean_module('03_model_runner.py', 'notebook_model_runner')
pinn, train_loader = runner.get_model_and_data()
device = runner.model_module.device
print(type(pinn).__name__, len(train_loader.dataset), device)

### 2.6 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| `_init_()` 写成 `__init__()` 覆盖父类方法 | 父类 `nn.Module.__init__()` 被跳过, 模型参数注册失败, `.parameters()` 返回空 | 用独立的方法名 `_init_()`, 两个下划线后缀 |
| Encoder 输出维度设太小 (如 8) | 嵌入空间不够表达 16 个特征的全部变化, SOH 预测不准 | 论文验证 32 维是最佳平衡 (Vapnik's principle) |
| Solution_u 的 `_init_` 忘记递归初始化子模块 | Encoder 和 Predictor 的 weight 使用 PyTorch 默认的均匀分布, 收敛慢 | 用 `self.modules()` 递归遍历所有子模块 |
| Predictor 的 `input_dim` 写成 40 | 实例化时传入 `input_dim=32` 覆盖默认值 → 没影响; 但如果忘记传参 → 40维输入但 Encoder 输出 32 维 → shape mismatch | 始终显式传入 `input_dim=32` |

---


<a id="sec3c"></a>
## 3. 3c: PINN 模型组装

> **对应 src/03_model.py 第 101-118 行**
> **对应原始代码 Model/Model.py 第 127-162 行**

一句话: `PINN.__init__()` 组装双模型 —— `Solution_u` (预测 SOH) 和 `dynamical_F` (描述退化动力学), 并配备损失函数 `MSELoss` 和 `ReLU`。


### 3.1 代码块


In [ ]:
def count_parameters(model):
    """统计模型可训练参数数量"""
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('The model has {} trainable parameters'.format(count))
    return count


class PINN(nn.Module):
    """物理信息神经网络：Solution_u + dynamical_F + autograd 自动微分"""
    def __init__(self, F_layers_num=3, F_hidden_dim=60):
        super(PINN, self).__init__()
        self.solution_u = Solution_u().to(device)
        self.dynamical_F = MLP(input_dim=35, output_dim=1,
                               layers_num=F_layers_num,
                               hidden_dim=F_hidden_dim,
                               dropout=0.2).to(device)
        self.loss_func = nn.MSELoss()
        self.relu = nn.ReLU()


### 3.2 🔍 逐行详解: PINN 模型组装

---

**【前置】count_parameters 工具函数**

```python
def count_parameters(model):                   # [函数定义头]
    #  └───────┬───────┘ └─┬──┘                   model: 任何 nn.Module 实例
    #     函数名(见名知义)  形参

    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    #       └┬┘ └───┬───┘ └┬┘ └──────┬──────┘ └┬┘ └─────┬─────┘
    #      [求和] [元素数] [循环] [model的所有参数] [过滤:需要梯度的]
    #
    #  逐段拆解:
    #  ┌────────────────────────────────────────────────────┐
    #  │ model.parameters() -> 返回模型所有参数的迭代器          │
    #  │ p.numel()          -> 返回参数 p 的元素总数             │
    #  │                      例如 Linear(17,50).weight: 850 │
    #  │ p.requires_grad    -> True=可训练, False=冻结         │
    #  │ if p.requires_grad -> 只统计需要梯度的参数               │
    #  │ sum(...)           -> 对所有参数的元素数求和             │
    #  └────────────────────────────────────────────────────┘
    #
    #  为什么用生成器表达式而不是列表推导式?
    #  sum(p.numel() for p ...)   <- 生成器, 不创建中间列表, 节省内存
    #  sum([p.numel() for p ...]) <- 列表推导式, 先创建列表再求和
    #  参数量大时(百万级)生成器更省内存
```

```python
    print('The model has {} trainable parameters'.format(count))
    #     [输出: format 填充 count 到 {} 位置]
    return count                               # [返回语句] 返回参数总数
    #  返回值让外部可以保存 n_params 用于日志
```

> **函数整体角色**: 纯工具函数 —— 输入模型, 输出参数量。不修改模型, 没有副作用。在 `__main__` 中用来验证模型结构。

---

**【模块 A】PINN 类的构造函数 —— 组装车间**

```python
class PINN(nn.Module):                         # [类定义头]
    #  └┬┘                                        PINN = Physics-Informed Neural Network
    #   类名                                        这个类是整个论文的核心
```

```python
    def __init__(self, F_layers_num=3, F_hidden_dim=60):
    #  [构造函数] [dynamical_F 的层数] [dynamical_F 的隐藏层宽度]
    #
    #  为什么只给 dynamical_F 提供配置参数, 不给 Solution_u?
    #  Solution_u 的结构由论文严格定义(17->60->60->32->1), 不改变
    #  dynamical_F 的结构可能在不同实验设置中调整(复杂度 vs 过拟合)
```

```python
        super(PINN, self).__init__()           # [调用父类构造函数]
        #  这一步让 PINN 继承 nn.Module 的 parameter 注册
        #  和 .to(device) / .eval() / .train() 等方法

        self.solution_u = Solution_u().to(device)
        #  └─────┬─────┘ └─────┬─────┘ └──┬──┘
        #   [属性赋值]  [实例化 Solution_u] [移到 GPU/CPU]
        #
        #  Solution_u = Encoder(MLP: 17->60->60->32) + Predictor(32->1)
        #  参数量: 7781
        #  角色: 给定 (特征, cycle_index), 预测 SOH
        #
        #  .to(device): 将模型的所有参数移到指定设备
        #  如果 device='cuda' → 模型在 GPU 上, 前向传播用 GPU 加速
        #  如果 device='cpu'  → 模型在 CPU 上
        #
        #  【关键】模型初始化和数据必须在同一设备上
        #  model.to('cuda') 但 data 在 cpu → RuntimeError
```

```python
        self.dynamical_F = MLP(input_dim=35, output_dim=1,
        #  └─────┬─────┘        [输入维度=35]  [输出维度=1]
        #   [属性赋值]
        #
        #  dynamical_F = 退化动力学函数 G(xt, u, u_x, u_t) -> 预测 u_t
        #  也就是 PDE 中的 G 函数: u_t = G(x, t, u, u_x)
                               layers_num=F_layers_num,
        #                      [层数=3] 第0层输入 + 第1层隐藏 + 第2层输出
                               hidden_dim=F_hidden_dim,
        #                      [隐藏层宽度=60]
                               dropout=0.2).to(device)
        #                      [Dropout率] [移到 GPU/CPU]
        #
        #  dynamical_F 参数估算 (layers_num=3, hidden_dim=60):
        #  第0层: 35*60 + 60 = 2160
        #  第1层: 60*60 + 60 = 3660
        #  第2层: 60*1 + 1 = 61
        #  合计: 2160 + 3660 + 61 = 5881
```

```python
        self.loss_func = nn.MSELoss()          # [MSE 损失函数实例]
        #  └────┬────┘ └────┬────┘                MSELoss = Mean Squared Error
        #   [属性赋值]  [实例化损失函数]             公式: MSE(y_pred, y_true) = mean((y_pred - y_true)^2)
        #
        #  为什么把损失函数放在模型里?
        #  方便: 训练时 self.loss_func(u1, y1) 直接调用
        #  L_data 和 L_PDE 都复用同一个 MSELoss 实例
        #  MSELoss 没有可训练参数, 不需要 .to(device)

        self.relu = nn.ReLU()                  # [ReLU 激活函数实例]
        #  └──┬──┘ └──┬──┘                         ReLU(x) = max(0, x)
        #  [属性赋值] [实例化激活]                     负值变0, 正值不变
        #
        #  为什么需要 ReLU? 它在模块 4 的 L_mono 中使用:
        #  L_mono = ReLU((u2-u1) * (y1-y2))
        #  含义: 只有当预测的 SOH 变化方向和真实变化方向相反时
        #  (即预测上升但真实下降, 或预测下降但真实上升)
        #  才产生惩罚。方向正确则不惩罚。
```

> **函数整体角色**: `PINN.__init__()` 是组装车间 —— 不定义新的计算逻辑, 只是把 `Solution_u` (预测 SOH) 和 `dynamical_F` (建模退化动力学) 两个子模型组合在一起, 并配备后续 Loss 计算需要的工具 (`MSELoss`, `ReLU`)。这是典型的**组合模式 (Composition)**: PINN 不继承 Solution_u, 而是拥有一个 Solution_u。

---

**【dynamical_F 的 35 维输入完整拆解】**

这是 PINN 中最容易被误解的设计。35 = 17 + 1 + 16 + 1, 每个维度的来源:

```
+=======================================================================+
|                    dynamical_F input (35 dims)                         |
+=======================================================================+
|  维度范围   | 长度 | 符号  | 内容              | 来源                 |
|------------|:----:|:-----:|-------------------|---------------------|
|  [0:17)    |  17  |  xt   | 原始输入向量       | DataLoader 直接提供   |
|            |      |       | = 16特征 + cycle  |                     |
|  [17:18)   |   1  |  u    | SOH 预测值         | Solution_u(xt)      |
|  [18:34)   |  16  |  u_x  | SOH 的空间梯度     | autograd.grad(u, x) |
|            |      |       | du/dx_i, i=1..16  |                     |
|  [34:35)   |   1  |  u_t  | SOH 的时间梯度     | autograd.grad(u, t) |
|            |      |       | du/dt (退化速率)   |                     |
+=======================================================================+
|  总计: 17 + 1 + 16 + 1 = 35                                          |
+=======================================================================+
```

G 的角色: 接收所有已知信息 → 学习如何预测 u_t (退化速率)
         → f = u_t - G(xt, u, u_x, u_t)
         → 如果 G 完美建模退化动力学, 则 f ~ 0
         → L_PDE = MSE(f, 0) 就是这个约束

注意: G 的输入中包含 u_t (时间梯度), 而 G 的目标也是预测 u_t。
这看起来是"用 u_t 预测 u_t"——但 G 看到的不只是 u_t, 还有 xt, u, u_x。
G 学习的是: "给定当前状态 (xt, u, u_x, u_t), 退化速率应该是多少?"
如果 G 精确建模了退化规律, 它的输出会与真实的 u_t 接近, 使得 f ~ 0。

---

**【关键设计决策】为什么 PINN 用组合而非继承?**

| 方案 | 代码 | 优点 | 缺点 |
|------|------|------|------|
| 继承 `class PINN(Solution_u)` | PINN 自动拥有 Solution_u 的全部方法 | ".to(device)" 对父类参数无效, 需额外处理 |
| **组合 (本实现)** | `self.solution_u = Solution_u()` | ✅ 清晰的边界; 子模型各自 .to(device); 易于替换组件 | 需显式调用 self.solution_u(x) |

组合模式让 PINN 的三个组件 (Solution_u, dynamical_F, autograd) 职责清晰、互不耦合。


### 3.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `.to(device)` | 将模型所有参数和 buffer 移到 GPU/CPU | 模型和张量必须在同一设备上才能运算; 初始化后立即调用 |
| `sum(p.numel() for p in model.parameters())` | 生成器表达式: 统计模型参数总数 | 比 `sum([...])` 节省内存(不创建中间列表) |
| `input_dim=35` | dynamical_F 的输入维度 | 17(xt) + 1(u) + 16(u_x) + 1(u_t) = 35 —— 必须精确匹配 |
| `nn.MSELoss()` | 均方误差损失: `mean((pred - target)^2)` | L_data 和 L_PDE 共用一个 MSELoss 实例, 节省内存 |
| `nn.ReLU()` | max(0, x): 负数变0, 正数不变 | L_mono 中只惩罚"预测变化方向与真实方向相反"的样本 |
| `F_layers_num` / `F_hidden_dim` | 参数名中的 `F_` 前缀 | 提示这些参数只控制 dynamical_F, 不控制 Solution_u |


### 3.4 逻辑: PINN 双模型的协作模式

```
                      +---------------------------+
                      |     PINN.forward(xt)      |
                      +-------------+-------------+
                                    |
              +---------------------+---------------------+
              |                     |                     |
              v                     v                     v
     +------------------+  +----------------+  +------------------+
     |  Solution_u      |  |  autograd      |  |  dynamical_F     |
     |  u = F(xt)       |  |  u_t = du/dt   |  |  G(xt,u,u_x,u_t) |
     |  SOH 预测        |  |  u_x = du/dx   |  |  退化动力学       |
     +--------+---------+  +-------+--------+  +--------+---------+
              |                    |                     |
              v                    v                     v
             u               u_t, u_x            f = u_t - G
         (参与 L_data)    (输入到 G)           (参与 L_PDE)
```

两个模型的定位完全不同:
- `Solution_u`: **正向预测** —— 给定特征和时间, SOH 是多少? (标准监督学习)
- `dynamical_F`: **物理约束** —— 退化速率应该满足什么规律? (PINN 特有)

如果只有 Solution_u 没有 dynamical_F → 退化为普通 MLP。如果没有 Solution_u 只有 dynamical_F → 无法预测 SOH。两者都必须有。


### 3.5 物理含义: PINN 如何对应 PDE?

Solution_u 和 dynamical_F 对应 PDE 中的两个角色:

$$\frac{\partial u}{\partial t} = \mathcal{G}(\mathbf{x}, t, u, \nabla_\mathbf{x} u)$$

| 数学符号 | 代码实现 | 含义 |
|----------|----------|------|
| $u$ (未知函数) | `self.solution_u(xt)` | SOH 作为时间和特征的函数 |
| $\frac{\partial u}{\partial t}$ (时间导数) | `grad(u.sum(), t)` | SOH 随循环的退化速率 |
| $\nabla_\mathbf{x} u$ (空间梯度) | `grad(u.sum(), x)` | SOH 对每个特征的敏感度 |
| $\mathcal{G}$ (退化动力学) | `self.dynamical_F(...)` | 从数据中学习到的退化方程 |
| $f = u_t - \mathcal{G}$ (PDE 残差) | `f = u_t - F` | 模型对物理规律的违反程度 |

PDE 残差 f = u_t - G 衡量神经网络对退化动力学建模的**一致性**: 如果 f ~ 0, 说明 Solution_u 的预测与退化规律自洽——模型不仅"记住了" SOH 值, 还"理解了"退化过程。

---

**【核心机制】自动微分的双层计算图**

```
第 1 层前向传播:
  xt --[solution_u]--> u --[grad(u.sum(), t)]--> u_t
                          --[grad(u.sum(), x)]--> u_x
                          --[dynamical_F]--> F
  f = u_t - F

第 2 层反向传播 (loss.backward()):
  L_PDE = MSE(f, 0)  --[backward 传播]--> dynamical_F 的权重
                                         --> solution_u 的权重 (经过 u_t, u_x)
```

`create_graph=True` 保证了第 2 层的 backward 能穿透第 1 层的 grad —— 这就是 PINN 比普通网络多一层计算图的代价和精髓。


### 3.6 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 忘记 `.to(device)` | `RuntimeError: Expected all tensors to be on the same device` | 模型初始化后立即 `.to(device)` |
| dynamical_F 的 `input_dim` 写错 | `mat1 and mat2 shapes cannot be multiplied (batch x 34 vs 35 x 60)` | 严格按 17+1+16+1=35 计算并验证 |
| 混淆 Solution_u 和 dynamical_F 的职责 | 丢失物理约束, 模型退化为普通 MLP | Solution_u=预测SOH, G=建模退化动力学, 不可互换 |
| 每个子模型单独 .to(device) | GPU 显存碎片化, 且繁琐 | 用 `self.to(device)` 替代, 或在 PINN 层面统一 .to(device) |

---


<a id="sec3d"></a>
## 4. 3d: PINN.forward() -- autograd 自动微分 [核心]

> **对应 src/03_model.py 第 120-139 行**
> **对应原始代码 Model/Model.py 第 216-235 行**

一句话: **这是整个 PINN 的心脏**。`forward()` 调用 `autograd.grad()` 引擎计算 $u_t = \partial u/\partial t$ 和 $u_x = \partial u/\partial x$, 再通过 `dynamical_F` 计算 PDE 残差 $f = u_t - \mathcal{G}$。


### 4.1 代码块


**代码结构片段（教学展示，不单独执行）**

```python
    def forward(self, xt):
        xt.requires_grad = True
        x = xt[:, 0:-1]
        t = xt[:, -1:]

        u = self.solution_u(torch.cat((x, t), dim=1))

        u_t = grad(u.sum(), t,
                   create_graph=True,
                   only_inputs=True,
                   allow_unused=True)[0]
        u_x = grad(u.sum(), x,
                   create_graph=True,
                   only_inputs=True,
                   allow_unused=True)[0]

        F = self.dynamical_F(torch.cat([xt, u, u_x, u_t], dim=1))

        f = u_t - F
        return u, f
```

### 4.2 🔍 逐行详解: PINN.forward() — 逐字符拆解

这是整个 PINN 项目**最重要、最精妙**的 20 行代码。下面逐行逐字符拆解, 包括每个参数的含义、每个张量的形状变化、以及每个设计决策的原因。

---

**【前置】函数签名与输入**

```python
    def forward(self, xt):                     # [前向传播定义]
        #  └──┬──┘ └┘ └┘                          xt: 输入张量, 来自 DataLoader
        #   方法名   self 形参                       形状: (batch, 17)
        #                                           17 = 16个特征 + 1个cycle_index
        #
        #  forward 是 nn.Module 的约定方法名 ——
        #  调用 model(x) 时, PyTorch 自动调用 model.forward(x)
```

---

**【步骤 1】启用梯度追踪 !==

```python
        xt.requires_grad = True                # [关键设置] 启用梯度追踪
        #  └┬┘ └─────┬─────┘
        #   Tensor  属性: 是否对 xt 计算梯度
        #
        #  逐段拆解:
        #  为什么必须设为 True?
        #  ┌────────────────────────────────────────────────────┐
        #  │ grad(u, t) 执行时, PyTorch 回溯计算图查找梯度路径      │
        #  │ 如果 xt.requires_grad=False (DataLoader 默认),       │
        #  │ 则 autograd 认为"不需要对 xt 求导" → t 也没有梯度      │
        #  │ → grad(u, t) 返回 None                            │
        #  │ 设为 True 后: xt 成为计算图的叶子节点                  │
        #  │ → t = xt[:, -1:] 继承了 requires_grad=True          │
        #  │ → grad(u, t) 正常工作                               │
        #  └────────────────────────────────────────────────────┘
        #
        #  代价: requires_grad=True 的张量占用更多显存
        #  (需要存储梯度信息), 但这是 PINN 的必要代价
```

---

**【步骤 2】分离特征和时间**

```python
        x = xt[:, 0:-1]                        # [切片分离] 取特征部分
        #  └┘ └──┬──┘ └┬┘ └┘
        #  赋值  原张量  所有行 取第0列到倒数第2列
        #
        #  形状: (batch, 17) -> (batch, 16)
        #  内容: 16个归一化后的统计特征
        #
        #  为什么用 0:-1 而不是 :-1 ?
        #  效果相同, 但 0:-1 更明确地表达"从第一列开始"
        #  :-1 是 "从开始到倒数第二列"

        t = xt[:, -1:]                         # [切片分离] 取时间部分
        #  └┘ └──┬──┘ └┘
        #  赋值  原张量  取最后一列
        #
        #  形状: (batch, 17) -> (batch, 1)  <- 注意保持 2D!
        #  内容: cycle_index (循环编号, 代表"时间")
        #
        #  【关键】为什么是 t = xt[:, -1:] 而不是 t = xt[:, -1] ?
        #  xt[:, -1]  -> 形状 (batch,)  <- 1D 向量
        #  xt[:, -1:] -> 形状 (batch, 1) <- 2D 列向量
        #  grad() 要求 input 至少 2D —— 1D 向量会报维度错误
```

---

**【步骤 3】SOH 预测**

```python
        u = self.solution_u(torch.cat((x, t), dim=1))
        #  └┘ └─────┬─────┘ └─────────┬────────┘ └─┬─┘
        #  赋值  Solution_u    torch.cat: 拼接    沿第1维
        #                       (列方向)
        #
        #  为什么先把 x 和 t 分开, 又立即拼回去?
        #  因为 autograd 需要知道哪一列是 t —— 通过分开-拼接来明确标记
        #  autograd 看到 t 是从 xt 的最后一列切出来的 → 可以追踪梯度
        #
        #  形状变化:
        #  cat((batch,16), (batch,1), dim=1) -> (batch, 17)
        #  solution_u(batch, 17) -> (batch, 1)
        #
        #  u 的物理含义: 预测的 SOH 值 (Health State of Charge)
```

---

**【步骤 4】自动微分: 计算 u_t 和 u_x**

```python
        u_t = grad(u.sum(), t,
        #  └┬┘ └──┬──┘ └┬┘ └┘
        #  输出  autograd  输出   对其求导
        #         .grad    标量   的输入
        #
        #  逐段拆解 grad() 的 5 个参数:
        #
        #  ┌─────────────────────────────────────────────────────────┐
        #  │ 参数1: outputs = u.sum()                                │
        #  │   u 形状是 (batch, 1), .sum() 后变成标量 (shape=())      │
        #  │   【关键】autograd.grad 要求 output 必须是标量!            │
        #  │   如果传 u (batch,1), 报错: grad can be implicitly       │
        #  │   created only for scalar outputs                       │
        #  │   u.sum() 把所有 batch 的 SOH 预测值加成一个标量          │
        #  │   等价于 "batch 内所有 SOH 的总体变化率"                  │
        #  ├─────────────────────────────────────────────────────────┤
        #  │ 参数2: inputs = t                                       │
        #  │   对 t 求导 —— 计算 du/dt                                │
        #  │   t 包含 cycle_index, 代表"时间"                        │
        #  ├─────────────────────────────────────────────────────────┤
        #  │ 参数3: create_graph=True   ← 【最关键参数】               │
        #  │   保留本次求导的计算图                                      │
        #  │   为什么? 因为后续 backward() 需要对 u_t 再求导            │
        #  │   如果 create_graph=False:                                │
        #  │     grad(u.sum(), t) 后计算图被释放                       │
        #  │     → backward() 无法通过 u_t 传播到 solution_u 的权重     │
        #  │     → L_PDE 的梯度为 0 → 物理约束失效!                   │
        #  │   这是 PINN 和普通 autograd 用法的最大区别!                │
        #  ├─────────────────────────────────────────────────────────┤
        #  │ 参数4: only_inputs=True                                  │
        #  │   只返回对 inputs 的梯度(不返回对中间变量的梯度)             │
        #  │   减少不必要的计算和内存                                  │
        #  ├─────────────────────────────────────────────────────────┤
        #  │ 参数5: allow_unused=True                                 │
        #  │   允许某些 input 在计算图中未被使用(不报错)                 │
        #  │   防御性编程: 如果 future 修改了 forward 逻辑, 不会因为     │
        #  │   某个 input 临时不用而崩溃                               │
        #  └─────────────────────────────────────────────────────────┘
        #
        #  返回值: grad() 返回一个 tuple, 每个元素对应一个 input
        #  [0] 取第一个(也是唯一一个)返回值 → u_t, 形状 (batch, 1)
        #
        #  物理含义: u_t = d(SOH)/d(cycle)
        #  负值 = SOH 下降(正常退化), 正值 = SOH 回升(容量恢复)
                   create_graph=True,
                   only_inputs=True,
                   allow_unused=True)[0]
        #                          └┬┘
        #                     [索引: 取tuple的第一个元素]
```

```python
        u_x = grad(u.sum(), x,
        #  对 x 求导 —— 计算 du/dx_i, i=1..16
        #  形状: (batch, 16) <- 对 16 列特征各出一个偏导数
        #  物理含义: SOH 对每个特征的敏感度
        #  如果某个特征的 du/dx_i 绝对值大 → 该特征对 SOH 影响大
        #  如果某个特征的 du/dx_i ~ 0 → 该特征几乎不影响 SOH
                   create_graph=True,
                   only_inputs=True,
                   allow_unused=True)[0]
```

---

**【步骤 5】退化动力学预测 + PDE 残差**

```python
        F = self.dynamical_F(torch.cat([xt, u, u_x, u_t], dim=1))
        #  └┘ └─────┬─────┘ └───────────────┬──────────────┘ └─┬─┘
        #  赋值  dynamical_F     torch.cat 拼接4个信息源    沿列方向
        #
        #  拼接的 4 个信息来源:
        #  xt:   (batch, 17) <- 原始输入 [0:17)
        #  u:    (batch, 1)  <- SOH 预测值 [17:18)
        #  u_x:  (batch, 16) <- 空间梯度 [18:34)
        #  u_t:  (batch, 1)  <- 时间梯度 [34:35)
        #  合计: (batch, 35) <- dynamical_F 的输入
        #
        #  形状变化: (batch, 35) -> MLP -> (batch, 1)
        #  F 的物理含义: G 对退化速率 u_t 的预测
        #  如果 G 完美建模退化动力学 → F ≈ u_t → f ~ 0
```

```python
        f = u_t - F                            # [PDE 残差计算]
        #  └┘ └┬┘ └┘
        #  残差 u_t   G的输出
        #
        #  形状: (batch, 1) — 每个样本的 PDE 残差
        #
        #  为什么 f = u_t - F 而不是 f = F - u_t ?
        #  两种写法数学上只差一个符号
        #  MSE(f, 0) = MSE(-f, 0), 所以符号无所谓
        #  但 u_t - F 的语义更清晰: "实际的退化速率 - 预测的退化速率"
        #  正值 = 实际退化比模型预测的更快, 负值 = 模型高估了退化速率
```

```python
        return u, f                            # [返回语句]
        #       └┘ └┘
        #       SOH PDE残差
        #
        #  u: (batch, 1) -> 参与 L_data = MSE(u, y_true)
        #  f: (batch, 1) -> 参与 L_PDE  = MSE(f, 0)
        #
        #  【关键】为什么 forward 返回两个值?
        #  普通网络 forward 只返回预测值
        #  PINN 多返回一个 PDE 残差 —— 这就是多出来的物理监督信号
```

> **函数整体角色**: `forward()` 是 PINN 的核心 —— 一次 forward, 同时得到预测值 `u` 和物理约束 `f`。`u` 参与数据损失 ($L_{data}$), `f` 参与物理损失 ($L_{PDE}$)。这 20 行代码是整个论文方法论的最小实现。

---

**【形状追踪表】forward() 全程张量形状变化**

| 变量 | 计算 | 形状 | 物理含义 |
|------|------|:---:|----------|
| `xt` | DataLoader 输入 | (B, 17) | 原始输入: 16特征 + cycle_index |
| `x` | `xt[:, 0:-1]` | (B, 16) | 16 个归一化特征 |
| `t` | `xt[:, -1:]` | (B, 1) | cycle_index (时间) |
| `u` | `solution_u([x,t])` | (B, 1) | SOH 预测值 |
| `u.sum()` | `u.sum()` | () 标量 | batch 内 SOH 总和 (grad 需要标量) |
| `u_t` | `grad(u.sum(), t)` | (B, 1) | 退化速率 dSOH/d(cycle) |
| `u_x` | `grad(u.sum(), x)` | (B, 16) | SOH 对每特征的敏感度 |
| `cat` | `cat([xt,u,u_x,u_t])` | (B, 35) | 拼接: 17+1+16+1 |
| `F` | `dynamical_F(cat)` | (B, 1) | G 预测的退化速率 |
| `f` | `u_t - F` | (B, 1) | PDE 残差: 接近 0 = 物理一致性 |

---

**【核心机制】create_graph=True 的计算图视角**

```
                         计算图边界
                    +--------------------+
                    |                     |
  xt ------------> solution_u ---> u ----+---> u_t (grad)   |
   |                    |               |                    |
   |                    |               +---> u_x (grad)   |
   |                    |               |                    |
   +--> cat([xt,u,u_x,u_t]) --> dynamical_F --> F           |
                                |                            |
                                +--> f = u_t - F            |
                                      |                      |
                                      v                      |
                                  L_PDE = MSE(f, 0)          |
                                      |                      |
                              backward()  ← 需要穿透 grad!   |
                                         +--------------------+

  create_graph=True: grad 之后计算图仍然存活
    -> backward() 能从 L_PDE 反向传播到 solution_u 的权重

  create_graph=False: grad 之后计算图被销毁
    -> backward() 在 grad 节点处被截断
    -> L_PDE 无法更新 solution_u -> PINN 退化为普通 MLP + 一个独立的 G 网络
```

**【直观理解】**: `create_graph=True` 相当于在自动微分的输出和原始计算图之间搭了一座桥。没有这座桥, `backward()` 走到了 `grad()` 这里就会被拦住 —— 就像一条路修到一半断了。有了这座桥, 梯度可以从 `f` 一直追溯到 `xt`, 再通过 `xt` 追溯到 `solution_u` 的所有权重。


### 4.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `xt.requires_grad = True` | 要求 PyTorch 对 xt 追踪梯度 | DataLoader Tensor 默认 `requires_grad=False`, 必须手动开启 |
| `xt[:, 0:-1]` / `xt[:, -1:]` | 按列切片: 分离特征和时间 | autograd 需要知道"对哪一列求导"(t列) |
| `u.sum()` | 将 (batch,1) 求和为标量 | `grad()` 要求输出为标量 —— 这是 PyTorch 的限制 |
| `grad(..., create_graph=True)` | 自动微分并保留计算图 | **PINN 的灵魂**: 让后续 backward 能穿透 grad 传播梯度 |
| `only_inputs=True` | 只返回对指定输入变量的梯度 | 减少不必要的中间变量梯度计算 |
| `allow_unused=True` | 允许某些输入在计算图中未被使用 | 防御性编程, 防止未来修改代码时崩溃 |
| `torch.cat([...], dim=1)` | 沿列方向拼接多个张量 | 构造 dynamical_F 的 35 维输入 |
| `.sum()` 用于 (B,1) 张量 | B 个样本求和 → 标量 | 对所有样本的 SOH 一视同仁地求导 |


### 4.4 逻辑: 为什么 forward 返回两个值?

```
              PINN.forward(xt)
                    |
        +-----------+-----------+
        v                       v
       u                       f
   SOH 预测值              PDE 残差
   (batch, 1)             (batch, 1)
        |                       |
        v                       v
  +-------------+      +-------------------+
  |   L_data    |      |     L_PDE         |
  | MSE(u, y)   |      |   MSE(f, 0)       |
  | "预测要准"   |      | "满足物理方程"     |
  +-------------+      +-------------------+
        |                       |
        +-----------+-----------+
                    |
                    v
            L_total = L_data + alpha*L_PDE + beta*L_mono
```

**如果 forward 只返回 u (像普通网络一样):**
- 只有 L_data → 退化为纯数据驱动的 MLP
- 失去了 PINN 的核心优势: 物理约束

**如果 forward 只返回 f:**
- 无法评估 SOH 预测的准确性
- L_data 无法计算

**所以必须返回两个。** u 和 f 各自监督不同方面的学习:
- u 受到真实 SOH 值的监督 (数据驱动)
- f 受到 PDE 方程的监督 (物理驱动)


### 4.5 物理含义: autograd 求的是真正的导数

用真实数据验证: 输入一个 batch → forward 输出 u 和 f 的分布:


In [ ]:
pinn.eval()
x_sample = None
for x1, _, _, _ in train_loader:
    x_sample = x1[:64].to(device)
    break

x_sample.requires_grad = True
u, f = pinn.forward(x_sample)

x_np = x_sample.detach().cpu().numpy()
u_np = u.detach().cpu().numpy()
f_np = f.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x_np[:, -1], u_np, c=u_np.flatten(), cmap='RdYlGn', s=20, alpha=0.7)
axes[0].set_xlabel('Cycle Index (t)')
axes[0].set_ylabel('Predicted SOH (u)')
axes[0].set_title('forward Output: SOH Prediction u')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(x_np[:, -1], f_np, c='blue', s=20, alpha=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, label='f=0 (ideal)')
axes[1].set_xlabel('Cycle Index (t)')
axes[1].set_ylabel('PDE Residual (f)')
axes[1].set_title('forward Output: PDE Residual f')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('PINN.forward() Output (Untrained Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'../outputs/figures/model/forward_output.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'u (SOH prediction) mean: {u_np.mean():.4f}, std: {u_np.std():.4f}')
print(f'f (PDE residual)  mean: {f_np.mean():.6f}, std: {f_np.std():.6f}')
print('Note: Untrained model f deviates far from 0 -> degradation dynamics not modeled')


### 4.6 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| `create_graph=False` (PyTorch 默认) | `RuntimeError: Trying to backward through the graph a second time` — grad() 后的图已被释放, backward() 找不到路径 | `create_graph=True` **必须设置**, 这是 PINN 的第一准则 |
| 忘记 `xt.requires_grad = True` | grad() 返回 None, 后续 cat 报错: `expected Tensor but got NoneType` | forward 开头第一件事: `xt.requires_grad = True` |
| `grad(u, t)` 而不是 `grad(u.sum(), t)` | `RuntimeError: grad can be implicitly created only for scalar outputs` | `.sum()` 将 (batch,1) 向量转为标量 |
| `t` 取成 1D: `xt[:, -1]` | grad() 维度不匹配: expected (batch,1) but got (batch,) | 用 `xt[:, -1:]` 保持 2D — 这是最容易犯的错误 |
| 多次 forward 后忘记 detach | 计算图累积, 显存溢出 | 推理时用 `torch.no_grad()`, 训练时 optimizer.step() 后 optimizer.zero_grad() |

---


<a id="sec3e"></a>
## 5. 3e: predict / Test / Valid

> **对应 src/03_model.py 第 141-172 行**
> **对应原始代码 Model/Model.py 第 183-214 行**

一句话: 在评估模式下做 SOH 预测 —— `predict()` 只走 Solution_u (不计算 PDE 残差), `Test()` 收集全部预测值和真值, `Valid()` 计算验证集 MSE 标量。


### 5.1 代码块


**代码结构片段（教学展示，不单独执行）**

```python
    def predict(self, xt):
        return self.solution_u(xt)

    def Test(self, testloader):
        self.eval()
        true_label = []
        pred_label = []
        with torch.no_grad():
            for iter, (x1, _, y1, _) in enumerate(testloader):
                x1 = x1.to(device)
                u1 = self.predict(x1)
                true_label.append(y1)
                pred_label.append(u1.cpu().detach().numpy())
        pred_label = np.concatenate(pred_label, axis=0)
        true_label = np.concatenate(true_label, axis=0)
        return true_label, pred_label

    def Valid(self, validloader):
        self.eval()
        true_label = []
        pred_label = []
        with torch.no_grad():
            for iter, (x1, _, y1, _) in enumerate(validloader):
                x1 = x1.to(device)
                u1 = self.predict(x1)
                true_label.append(y1)
                pred_label.append(u1.cpu().detach().numpy())
        pred_label = np.concatenate(pred_label, axis=0)
        true_label = np.concatenate(true_label, axis=0)
        mse = self.loss_func(torch.tensor(pred_label),
                             torch.tensor(true_label))
        return mse.item()
```

### 5.2 🔍 逐行详解: predict / Test / Valid

---

**【模块 A】predict —— 最简单的推理入口**

```python
    def predict(self, xt):                     # [推理方法]
        #  └──┬──┘ └┘ └┘                          predict = 预测, 与 forward 对仗
        #   方法名  self 形参                        xt: 输入张量, 形状 (batch, 17)

        return self.solution_u(xt)             # [返回语句: 直接调用 Solution_u]
        #       └─────┬─────┘
        #      Solution_u.forward(xt)
        #
        #  与 forward() 的核心区别:
        #  ┌──────────────────────────────────────────────────────┐
        #  │              forward()           predict()           │
        #  │ 返回值       (u, f) 两个值         u 一个值            │
        #  │ autograd     计算 u_t, u_x        不计算              │
        #  │ dynamical_F  调用 G 算 PDE 残差    不调用              │
        #  │ 用途         训练时计算 Loss       推理时只预测 SOH     │
        #  │ 速度         慢 (做自动微分)        快 (纯前向)         │
        #  │ 显存占用     高 (存计算图)          低 (不存图)         │
        #  └──────────────────────────────────────────────────────┘
        #
        #  推理时不需要 PDE 残差 —— 只有训练时 L_PDE 才需要 f
        #  推理时只需要 "这个电池现在的 SOH 预测是多少?"
```

> **predict 的角色**: 最轻量的推理入口 —— 不走 autograd, 不走 dynamical_F, 只走 Solution_u。类比: `forward()` 是"做实验"(需要测量各种物理量), `predict()` 是"回答问题"(只给结论)。

---

**【模块 B】Test —— 完整测试集评估**

```python
    def Test(self, testloader):                # [测试方法]
        #  └┬┘ └─────┬─────┘                     testloader: DataLoader 实例
        #   方法名   形参                           每个 batch yield (x1, x2, y1, y2)
        #                                          Test 只用 x1 和 y1, 丢弃 x2 和 y2
```

```python
        self.eval()                            # [切换评估模式]
        #  └──┬──┘                                nn.Module.eval(): 设置所有子模块为 eval 模式
        #  方法调用
        #
        #  评估模式 vs 训练模式的关键区别:
        #  ┌────────────────┬──────────────┬──────────────┐
        #  │  层类型         │ model.train()│ model.eval() │
        #  ├────────────────┼──────────────┼──────────────┤
        #  │ Dropout        │ 随机丢弃 20% │ 所有神经元参与 │
        #  │ BatchNorm      │ 用 batch 统计│ 用全局统计    │
        #  │ (本项目无 BN)   │ 量更新 running│ 量(不更新)    │
        #  └────────────────┴──────────────┴──────────────┘
        #
        #  【关键】如果测试时不调 self.eval():
        #  Dropout 仍然随机屏蔽神经元 → 每次测试结果不同!
        #  同一个模型同一组数据, 跑两次得到不同预测值 → 不可复现
```

```python
        true_label = []                        # [变量初始化] 收集所有 batch 的真值
        pred_label = []                        # [变量初始化] 收集所有 batch 的预测值
        #  为什么用列表? 每个 batch 的样本数可能不同
        #  (最后一个 batch 可能不满 batch_size)
        #  先用列表收集, 最后一次性 np.concatenate 拼接
```

```python
        with torch.no_grad():                  # [禁用梯度计算]
        #    └────┬────┘                         上下文管理器: 代码块内不追踪梯度
        #  torch.no_grad()
        #
        #  为什么测试需要 torch.no_grad()?
        #  ┌────────────────────────────────────────────────────┐
        #  │ 1. 节省显存: 不需要存中间激活值(activation)          │
        #  │    训练时存激活值 → backward 用; 测试时不 backward    │
        #  │    显存节省 ~30-50%                                │
        #  │ 2. 加速推理: 不构建计算图 → 前向传播更快              │
        #  │    速度提升 ~20-30%                                │
        #  │ 3. 防止误操作: 即使意外调用了 backward 也不会出错      │
        #  │    避免"测试时不小心改变模型权重"的错误               │
        #  └────────────────────────────────────────────────────┘
```

```python
            for iter, (x1, _, y1, _) in enumerate(testloader):
            #  └┬┘  └──┬──┘ └┘ └┘ └┘ └┘ └──────────┬──────────┘
            #  循环   枚举   解包       丢弃        迭代 DataLoader
            #  变量  (index, batch)  x2, y2
            #
            #  逐段拆解:
            #  enumerate(testloader) -> 返回 (序号, batch) 的迭代器
            #  iter: 当前是第几个 batch (0, 1, 2, ...)
            #  batch: DataLoader 产出的四元组 (x1, x2, y1, y2)
            #  (x1, _, y1, _): 序列解包
            #    x1: 第一个时刻的特征 (batch, 17)
            #    _:  丢弃 x2 (测试不需要第二个时刻)
            #    y1: 第一个时刻的真值 SOH (batch, 1)
            #    _:  丢弃 y2 (测试不需要)
            #
            #  为什么测试只需要 x1 和 y1?
            #  Test() 只关心 SOH 预测的准确性
            #  x1 里有完整特征 + cycle_index → 足以预测 SOH
            #  y1 用来和预测值对比算指标
            #  x2 和 y2 是为 L_PDE 和 L_mono 准备的(训练时用)
```

```python
                x1 = x1.to(device)             # [移到 GPU/CPU]
                #    └────┬────┘                   确保张量和模型在同一设备
                #   Tensor.to(device)

                u1 = self.predict(x1)          # [推理调用]
                #    └───┬───┘                     predict(): 只走 Solution_u, 不算 PDE
                #   调用 predict 方法                 返回 (batch, 1) 的 SOH 预测值

                true_label.append(y1)           # [列表追加] 收集真值
                #  y1 是 Tensor, 形状 (batch, 1), 仍在 GPU/CPU 上

                pred_label.append(u1.cpu().detach().numpy())
                #               └┬┘ └──┬──┘ └──┬──┘
                #              GPU→CPU 脱离图 转NumPy
                #
                #  三步拆解:
                #  ┌──────────────────────────────────────────────────┐
                #  │ .cpu():   把 Tensor 从 GPU 显存复制到 CPU 内存     │
                #  │           如果已经在 CPU 上, 无操作(no-op)          │
                #  │ .detach(): 从计算图中分离 → 创建一个新 Tensor       │
                #  │           不共享梯度, 不需要 backward 追踪          │
                #  │ .numpy():  PyTorch Tensor → NumPy ndarray        │
                #  │           必须是 CPU 上的 Tensor (GPU Tensor 不行)  │
                #  │           必须在 detach() 之后 (require_grad 的不行) │
                #  └──────────────────────────────────────────────────┘
                #
                #  为什么需要这三步?
                #  最终要返回 NumPy 数组给下游(sklearn, matplotlib)
                #  不能直接传 GPU PyTorch Tensor → sklearn 不认识
```

```python
        pred_label = np.concatenate(pred_label, axis=0)
        #           └─────┬─────┘ └────┬────┘ └─┬─┘
        #         [numpy 拼接]   [列表]    [沿行方向]
        #
        #  每个 batch 的 pred_label 形状: (batch_size, 1) 或 (last_batch, 1)
        #  concatenate 后: (N, 1) ← N = 测试集总样本数
        #
        true_label = np.concatenate(true_label, axis=0)
        #  同上: 拼接成 (N,) 或 (N, 1)
        #  返回的 true_label 和 pred_label 形状一致

        return true_label, pred_label           # [返回语句]
        #       └────┬────┘ └────┬────┘
        #       真实 SOH 值    预测 SOH 值
        #       形状 (N,)      形状 (N, 1) 或 (N,)
        #
        #  返回完整数组 → 下游可以用任意指标评估:
        #  MAE, RMSE, MAPE, R^2, 退化曲线图, 残差分布图, ...
```

> **Test 的整体角色**: 一次遍历整个测试集 → 收集所有预测值和真值 → 返回两个完整数组。不计算 MSE (留待模块 6), 只做"收集"。

---

**【模块 C】Valid —— 验证集 MSE 计算**

```python
    def Valid(self, validloader):              # [验证方法]
        self.eval()                            # [评估模式] 与 Test 完全相同
        true_label = []
        pred_label = []
        with torch.no_grad():                  # [禁用梯度] 与 Test 完全相同
            for iter, (x1, _, y1, _) in enumerate(validloader):
                x1 = x1.to(device)
                u1 = self.predict(x1)
                true_label.append(y1)
                pred_label.append(u1.cpu().detach().numpy())
        #  以上 9 行: 收集逻辑与 Test 完全一致

        pred_label = np.concatenate(pred_label, axis=0)
        true_label = np.concatenate(true_label, axis=0)

        mse = self.loss_func(torch.tensor(pred_label),
        #    └────┬────┘ └──────────┬──────────┘
        #   复用 PINN 的 MSELoss    NumPy -> PyTorch Tensor
        #    (在 __init__ 中创建)     (从 CPU NumPy 转回 Tensor)
                             torch.tensor(true_label))
        #   MSE = mean((pred - true)^2)
        #   返回一个标量 Tensor, 形状 ()

        return mse.item()                      # [Tensor 标量 -> Python float]
        #       └───┬───┘                          .item(): 将 0-dim Tensor 转为 Python float
        #      取 Python 数字                         方便日志记录, 不需要再 .detach()
```

> **Valid 的角色**: 与 Test 收集逻辑相同, 但多了一步 —— 用 `self.loss_func` 计算 MSE, 返回一个标量。训练循环中每个 epoch 调一次, 用于判断 early stop。

---

**【Test vs Valid vs predict 对比总结】**

| 特性 | `predict()` | `Test()` | `Valid()` |
|------|:-----------:|:--------:|:---------:|
| 返回值 | u (Tensor) | (true, pred) NumPy | mse (float) |
| 是否算 PDE? | ❌ 不走 autograd | ❌ 不走 autograd | ❌ 不走 autograd |
| 设置 eval 模式? | ❌ 调用者负责 | ✅ 内部调用 | ✅ 内部调用 |
| torch.no_grad()? | ❌ 调用者负责 | ✅ 内部包裹 | ✅ 内部包裹 |
| 遍历整个数据集? | ❌ 单个 batch | ✅ 所有 batch | ✅ 所有 batch |
| 用途 | 随时快速推理 | 最终测试报告 | 训练中验证 |

---

**【状态切换总结】各方法对 model 模式的影响**

| 方法 | model mode | torch.no_grad() | Dropout 行为 | 显存占用 | 何时调用 |
|------|:----------:|:---------------:|:------------:|:--------:|----------|
| `forward()` | training | No | 随机丢弃 20% | 高 (存计算图) | 训练时每个 batch |
| `predict()` | 不改变 | 不包含 | 取决于外部 | 取决于外部 | 随时可用 |
| `Test()` | eval | Yes | 全神经元参与 | 低 (不存图) | 每个 epoch 结束 |
| `Valid()` | eval | Yes | 全神经元参与 | 低 (不存图) | 每个 epoch 结束 |

---

**【形状追踪】Test/Valid 全过程**

```
testloader 有 M 个 batch:
  batch 0: x1 (256,17) -> predict -> u1 (256,1) -> .cpu().detach().numpy() -> (256,1) array
  batch 1: x1 (256,17) -> predict -> u1 (256,1) -> ... -> (256,1) array
  ...
  batch M-1: x1 (K,17) -> predict -> u1 (K,1) -> ... -> (K,1) array  (K <= 256)

  所有 batch 的 NumPy 数组 lst = [(256,1), (256,1), ..., (K,1)]
  np.concatenate(lst, axis=0) -> (N, 1) array
  N = 256*(M-1) + K = 测试集总样本数
```


### 5.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `self.eval()` | 切换模型和所有子模块到评估模式 | Dropout 在训练时随机屏蔽神经元, 评估时必须全部神经元参与 |
| `torch.no_grad()` | 上下文管理器: 禁用梯度计算 | 测试不需要反向传播 → 节省显存 ~30-50%, 加速 ~20-30% |
| `.cpu().detach().numpy()` | GPU→CPU → 脱离计算图 → 转 NumPy | PyTorch Tensor 不能直接传给 sklearn/matplotlib |
| `np.concatenate(list, axis=0)` | 沿第 0 维(行)拼接多个数组 | 不同 batch 的预测结果拼成完整测试集输出 |
| `.item()` | 0-dim Tensor 标量 → Python float | 方便日志记录和比较, 不需要 .detach() |
| `(x1, _, y1, _) = batch` | 序列解包: `_` = 丢弃不需要的元素 | Test/Valid 只需要 x1 (特征) 和 y1 (真值), 不需要配对信息 |
| `enumerate(dataloader)` | 迭代 DataLoader 同时返回序号 | iter 变量虽未使用但 enumerate 是惯用法; 也可用 `for batch in loader` |


### 5.4 逻辑: 推理 vs 训练的代码路径对比

```
训练模式 (train_one_epoch, 模块 5):
  for x1, x2, y1, y2 in train_loader:      <- 需要完整的配对数据 (x1,x2)
      u1, f1 = forward(x1)                  <- 计算 PDE 残差 (走 autograd + dynamical_F)
      u2, f2 = forward(x2)                  <- 计算 PDE 残差 (走 autograd + dynamical_F)
      loss = L_data + L_PDE + L_mono       <- 三项 Loss: 数据+物理+单调性
      loss.backward()                       <- 反向传播: 更新 Solution_u 和 dynamical_F
      optimizer.step()

推理模式 (Test / Valid):
  for x1, _, y1, _ in loader:               <- 只取 x1 和 y1, 丢弃 x2 和 y2
      u1 = predict(x1)                      <- 只算 SOH, 不计算 PDE 残差
      collect(u1, y1)                       <- 收集预测值和真值

  没有 backward, 没有 optimizer, 模型权重不变
```

**为什么推理可以丢弃 x2?**
- 训练需要 (x1, x2) 配对是因为 L_PDE 依赖 u_t (需要知道"从 x1 到 x2 SOH 变了多少")
- 推理只需要 SOH 预测值 —— 一个时刻就足够
- 不需要计算退化速率 u_t → 不需要第二个时刻 → 不需要 x2
- 不需要验证单调性 → 不需要比较 u2 和 u1 → 不需要 x2


### 5.5 物理含义: 测试输出代表什么?

`Test()` 输出的是**预测 SOH vs 真实 SOH 的完整序列** —— 这是评估电池退化预测准确性的核心数据。

下游模块 6 会基于 `(true_label, pred_label)` 这两个数组计算:
- **MAE** (平均绝对误差): `mean(|pred - true|)` → 平均每个循环预测差多少 SOH
- **RMSE** (均方根误差): `sqrt(mean((pred-true)^2))` → 对大误差更敏感
- **退化曲线对比**: 逐循环画出 pred vs true SOH 曲线
- **残差分布**: pred - true 的直方图 → 检查是否有系统性偏差

注意: 未训练的模型 (随机权重) 的预测是随机的 —— 只有训练后 Test 才有意义。


### 5.6 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| Test 时忘记 `self.eval()` | Dropout 仍然随机屏蔽神经元 → 每次测试结果不同 → 不可复现 | `Test()` 和 `Valid()` 开头必须 `self.eval()` |
| 不包裹 `torch.no_grad()` | 显存占用翻倍, 推理速度慢 2-3 倍; 且可能意外修改权重 | 始终在 `with torch.no_grad()` 下推理 |
| 测试集用了 `shuffle=True` 的 DataLoader | 预测值和真值的顺序不对应 → 退化曲线乱序 → 看起来像"模型完全不行" | 测试 DataLoader 创建时 `shuffle=False` |
| `.numpy()` 在 `.detach()` 之前 | `RuntimeError: Can't call numpy() on Tensor that requires grad` | 先 `.detach()` 脱离计算图, 再 `.numpy()` |
| `.numpy()` 在 `.cpu()` 之前 | `TypeError: can't convert cuda:0 device type tensor to numpy` | GPU Tensor 必须先 `.cpu()` 移到 CPU |

---


<a id="sec6"></a>
## 6. 主程序验证

> **对应 src/03_model.py 第 175-212 行**

一句话: 实例化 PINN 模型, 统计参数量, 随机输入跑一次 forward, 验证模型结构正确。这是整个模块的**集成测试**。


### 6.1 代码块


In [ ]:
import sys
from pathlib import Path
import torch
CLEAN = Path(r'../src')
sys.path.insert(0, str(CLEAN))
from module_loader import load_clean_module
model_module = load_clean_module('03_model.py', 'notebook_model_verify')
pinn = model_module.PINN()
x = torch.randn(4, 17, device=model_module.device)
u, residual = pinn(x)
print(tuple(u.shape), tuple(residual.shape))

### 6.2 🔍 逐行详解: 主程序验证

这是笔记本的最终执行单元 —— 把前面所有类 (Sin, MLP, Predictor, Solution_u, PINN) 串起来, 跑一次完整的模型架构验证。

---

```python
if __name__ == '__main__':
#  └────────────┬────────────┘
#     [Python 特殊变量检查]
#     含义: 只有当这个文件被直接运行时 (python 03_model.py),
#     才执行下面的代码。如果被 import (如 import 03_model),
#     则跳过 —— 防止 import 时误执行验证程序。
```

```python
    print("=" * 70)
    print("PINN4SOH 模块 3: PINN 前向传播 — 模型架构验证")
    print("=" * 70)
    # [输出标题] "=" * 70 = 70 个等号组成分隔线
```

---

**【阶段 A】模型实例化**

```python
    pinn = PINN(F_layers_num=3, F_hidden_dim=60)
    #     └┬┘ └─────┬─────┘ └─────┬─────┘
    #   实例化  dynamical_F层数 隐藏层宽度
    #
    #  内部发生了什么?
    #  PINN.__init__() 被调用:
    #  1. super().__init__() -> 注册 nn.Module
    #  2. self.solution_u = Solution_u().to(device)
    #     -> Encoder(MLP 17->60->60->32) + Predictor(32->1)
    #     -> 7781 个参数
    #  3. self.dynamical_F = MLP(35->60->60->1).to(device)
    #     -> 5881 个参数
    #  4. self.loss_func = nn.MSELoss()
    #  5. self.relu = nn.ReLU()
    #  PINN 总参数: 7781 + 5881 = 13662
```

---

**【阶段 B】参数统计**

```python
    print("Solution_u 结构:")
    print(pinn.solution_u)                     # [打印模型结构]
    #  输出: Solution_u 的层结构文本表示
    #  PyTorch 的 print(nn.Module) 会显示子模块树形图

    count_parameters(pinn.solution_u)          # [统计参数] 预期输出: 7781
    #  验证 Encoder + Predictor 的参数量符合预期

    print("dynamical_F 结构:")
    print(pinn.dynamical_F)

    count_parameters(pinn.dynamical_F)         # [统计参数] 预期输出: 5881
    #  如果输出不对 → 检查 input_dim=35 是否写错
    #  如果少了 → 某层没正确构建
```

---

**【阶段 C】forward 验证**

```python
    batch_size = 4
    n_features = 17
    x_test = torch.randn(batch_size, n_features).to(device)
    #       └─────┬─────┘ └────┬────┘ └──┬──┘ └──┬──┘
    #    [标准正态分布]   [4个样本] [17列] [移到GPU]
    #
    #  torch.randn: 从标准正态分布 N(0,1) 随机采样
    #  为什么用随机输入? 模型还没训练, 不需要真实数据
    #  只是验证"输入能走通, 输出形状正确"

    u, f = pinn.forward(x_test)
    # └┘ └┘ └────┬────┘ └─┬──┘
    # 输出 输出 调用forward 输入
    #
    #  调用 PINN.forward(x_test):
    #  1. xt.requires_grad = True
    #  2. x = xt[:, 0:-1], t = xt[:, -1:]
    #  3. u = solution_u(cat(x, t))  -> (4, 1)
    #  4. u_t = grad(u.sum(), t)     -> (4, 1)
    #  5. u_x = grad(u.sum(), x)     -> (4, 16)
    #  6. F = dynamical_F(cat([xt,u,u_x,u_t])) -> (4, 1)
    #  7. f = u_t - F                -> (4, 1)
    #  8. return u, f

    print("单次 forward 验证:")
    print("  输入形状: {}".format(list(x_test.shape)))
    #  预期: [4, 17]

    print("  u (SOH 预测值) 形状: {}".format(list(u.shape)))
    #  预期: [4, 1] — 4 个样本各一个 SOH 预测值

    print("  f (PDE 残差)   形状: {}".format(list(f.shape)))
    #  预期: [4, 1] — 4 个样本各一个 PDE 残差

    print("  u 值范围: [{:.4f}, {:.4f}]".format(
        u.min().item(), u.max().item()))
    #  u 的范围: 未训练模型, 值可能在任何范围
    #  训练后应该 ∈ [0.8, 1.0]

    print("  f 值范围: [{:.6f}, {:.6f}]".format(
        f.min().item(), f.max().item()))
    #  f 的范围: 未训练模型, 离 0 很远
    #  训练后应该逐渐接近 0 (L_PDE 的优化目标)

    print("模块 3 验证通过。")
    # 到这里没报错 → 模型结构正确 → 可以进入模块 4 (Loss 设计)
```

> **整体角色**: 这个 `if __name__ == '__main__':` 块是整个模块的集成测试 —— 不需要外部调用, 直接 `python 03_model.py` 就能验证模型结构是否正确。如果这里报错, 说明前面的类定义有问题; 如果通过, 说明可以安全进入模块 4。

---

**【主程序与函数的对应关系】**

```
主程序执行流程                         对应的函数/类
─────────────────────────────────────────────────────
pinn = PINN(...)                       → PINN.__init__()
  └─ Solution_u()                       → Solution_u.__init__()
       └─ MLP(17->32)                   → MLP.__init__()
            └─ for i in range(3) ...    → 构建 3 层网络
       └─ Predictor(32)                 → Predictor.__init__()
  └─ MLP(35->1)                         → MLP.__init__()
       └─ for i in range(3) ...         → 构建 3 层网络
count_parameters(solution_u)            → count_parameters()
count_parameters(dynamical_F)           → count_parameters()
pinn.forward(x_test)                    → PINN.forward()
  └─ solution_u(cat(x,t))              → Solution_u.forward()
  └─ grad(u.sum(), t)                  → autograd.grad()
  └─ grad(u.sum(), x)                  → autograd.grad()
  └─ dynamical_F(cat([...]))           → MLP.forward()
```


### 6.3 语法速查表

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `if __name__ == '__main__':` | Python 标准入口守卫 | 直接运行才执行, import 不执行 |
| `"=" * 70` | 字符串重复 70 次 | 生成分隔线 |
| `torch.randn(B, D)` | 从 N(0,1) 随机采样 (B,D) 形状 | 验证不需要真实数据, 随机张量足以验证形状 |
| `.to(device)` | 张量移到 GPU/CPU | 与模型在同一设备才能 forward |
| `list(tensor.shape)` | 将 torch.Size 转为 Python list | 方便 print 显示 |
| `.min()`, `.max()` | 返回 Tensor 的最小/最大值 | 检查 SOH 和 PDE 残差的数值范围 |
| `.item()` | 0-dim Tensor → Python float | 让 print 输出更简洁 |


### 6.4 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 在 GPU 上创建 Tensor 但不 .to(device) 模型 | cuda Tensor + cpu model → RuntimeError | 模型和输入数据必须在同一设备 |
| 忘记 main guard | 被其他脚本 import 时自动执行验证 → 产生不需要的输出 | 始终用 `if __name__ == '__main__':` 保护 |
| `batch_size` 太大 → 随机输入也爆显存 | CUDA out of memory | 验证用 batch_size=4 就够, 不需要 256 |

---


## 模块 3 总结: PINN 前向传播完整调用链

```
+------------------------------------------------------------------+
|                    PINN Forward Pass Complete Flow                 |
+------------------------------------------------------------------+
|                                                                  |
|  DataLoader -> (x1, x2, y1, y2) batch                            |
|       |                                                          |
|       +-- u1, f1 = pinn.forward(x1)                               |
|       |    +-- x = x1[:,0:-1] (16 features), t = x1[:,-1:] (time)|
|       |    +-- u1 = Solution_u.forward([x, t])                    |
|       |    +-- u_t = grad(u1.sum(), t)    <- autograd             |
|       |    +-- u_x = grad(u1.sum(), x)    <- autograd             |
|       |    +-- F1 = dynamical_F([x, t, u1, u_x, u_t])            |
|       |    +-- f1 = u_t - F1                                      |
|       |                                                          |
|       +-- u2, f2 = pinn.forward(x2)  <- same flow                |
|                                                                  |
|  Output:                                                          |
|    u1, u2 -> L_data = 0.5*MSE(u1,y1) + 0.5*MSE(u2,y2)            |
|    f1, f2 -> L_PDE  = 0.5*MSE(f1,0)  + 0.5*MSE(f2,0)            |
|    u1, u2 -> L_mono = ReLU((u2-u1)*(y1-y2))                     |
|                                                                  |
+------------------------------------------------------------------+
```

### 模型架构汇总

| 组件 | 类 | 输入 -> 输出 | 参数量 | 激活 |
|------|-----|------------|:------:|:----:|
| Sin 激活 | `Sin` | x -> sin(x) | 0 | sin |
| MLP 构造器 | `MLP` | (batch,N) -> (batch,M) | 可变 | sin |
| Encoder | `MLP(17->60->60->32)` | (batch,17) -> (batch,32) | 6,692 | sin |
| Predictor | `Predictor(32->1)` | (batch,32) -> (batch,1) | 1,089 | sin |
| **Solution_u** | `Solution_u` | (batch,17) -> (batch,1) | **7,781** | sin |
| **dynamical_F** | `MLP(35->60->60->1)` | (batch,35) -> (batch,1) | **5,881** | sin |
| **PINN** | `PINN` | (batch,17) -> (batch,1),(batch,1) | **13,662** | sin+ReLU |

### 数据流全景 (模块 1-3)

```
[模块1] .mat -> 特征提取 -> CSV (18列)
                                |
[模块2]                         v
        3sigma清洗 -> SOH转换 -> min-max归一化 -> 时间步配对 (x1,x2)
                                |
                                v
                      DataLoader (shuffle)
                                |
[模块3]                         v
                      PINN.forward(x1) --> (u1, f1)
                      PINN.forward(x2) --> (u2, f2)
                                |
                                v
[模块4]                三项 Loss 计算
```


### 与原论文的已知差异

| 差异项 | 程度 | 原因 |
|--------|------|------|
| Predictor 默认 `input_dim=40` 改为 `32` | 无影响 | 实际调用时传入 `input_dim=32` (Encoder 输出维度) |
| `PINN.__init__` 不包含 optimizer/scheduler | 架构分离 | 模块 3 只负责前向传播, 训练逻辑在模块 5 |
| 去除了 `log_dir/save_folder` 等日志参数 | 代码简化 | 训练脚手架在模块 5 中独立管理 |
| 去除了 `pi` 和 `Tanh` 等辅助组件 | 代码简化 | 论文复杂版本的功能, 核心复现不需要 |

---
